In [ ]:
import os

print("[*] Checking for existing Intel oneAPI installation...")
has_oneapi = os.path.exists("/opt/intel/oneapi")

if not has_oneapi:
    print("[*] Configuring Intel Official Package Repository...")
    commands = '''
    set -e
    wget -qO - https://apt.repos.intel.com/intel-gpg-keys/GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB | gpg --dearmor | sudo tee /usr/share/keyrings/oneapi-archive-keyring.gpg > /dev/null
    echo "deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main" | sudo tee /etc/apt/sources.list.d/oneAPI.list
    sudo apt-get update -q
    # Install standalone oneAPI DPC++/C++ compiler and MPI developer headers
    sudo apt-get install -y -q intel-oneapi-compiler-dpcpp-cpp intel-oneapi-mpi-devel build-essential
    '''
    # We execute setup in background / check readiness
    print("[+] Intel repository commands prepared.")
else:
    print("[+] Intel oneAPI is already installed at /opt/intel/oneapi!")

[*] Checking for existing Intel oneAPI installation...
[*] Configuring Intel Official Package Repository...
[+] Intel repository commands prepared.


In [ ]:
import subprocess

# Execute the prepared installation commands in the shell environment
if not has_oneapi:
    print("[*] Installing Intel oneAPI packages (this might take a few minutes)... ")
    process = subprocess.Popen(commands, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    # Stream output in real-time
    for line in iter(process.stdout.readline, ''):
        print(line, end='')

    process.stdout.close()
    return_code = process.wait()
    if return_code == 0:
        print("\n[+] Intel oneAPI environment installed successfully! Please re-run the compilation cells.")
    else:
        print(f"\n[!] Installation failed with exit code: {return_code}")
else:
    print("[+] Intel oneAPI is already available.")

[*] Installing Intel oneAPI packages (this might take a few minutes)... 
deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://apt.repos.intel.com/oneapi all InRelease [5,684 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1,578 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 https://apt.repos.intel.com/oneapi all/main amd64 Packages [960 kB]
Get:7 https://apt.repos.intel.com/oneapi all/main all Packages [270 kB]
Hit:8 http://archive.ubuntu.com/ubuntu noble InRelease
Get:9 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [78.0 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]


In [ ]:
import os
os.makedirs('/opt/intel/oneapi', exist_ok=True)

# Create a mock/redirecting setvars.sh that exports paths to icpx and system-installed oneAPI tools
setvars_content = """#!/bin/bash
# Automatically exported configurations for oneAPI components
export PATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin
export PATH=$PATH:$(dirname $(which icpx 2>/dev/null || echo '/usr/bin'))
echo ':: oneAPI environment initialized manually ::'
"""

with open('/opt/intel/oneapi/setvars.sh', 'w') as f:
    f.write(setvars_content)

print('[+] Successfully generated /opt/intel/oneapi/setvars.sh')

[+] Successfully generated /opt/intel/oneapi/setvars.sh


In [ ]:
!bash -c "source /opt/intel/oneapi/setvars.sh && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query"


:: oneAPI environment initialized manually ::
bash: line 1: icpx: command not found


In [ ]:
import os
import subprocess

# Let's locate the actual vars.sh or icpx path under /opt/intel/oneapi
print("Searching for icpx or setvars in /opt/intel...")
paths = []
for root, dirs, files in os.walk("/opt/intel"):
    for file in files:
        if file in ["setvars.sh", "vars.sh", "icpx"]:
            paths.append(os.path.join(root, file))

print("Found matching files:")
for p in paths:
    print(p)

Searching for icpx or setvars in /opt/intel...
Found matching files:
/opt/intel/oneapi/setvars.sh
/opt/intel/oneapi/libfabric/2.6/env/vars.sh
/opt/intel/oneapi/libfabric/2.6/etc/libfabric/vars.sh
/opt/intel/oneapi/umf/1.2/env/vars.sh
/opt/intel/oneapi/debugger/2026.2/env/vars.sh
/opt/intel/oneapi/debugger/2026.2/etc/debugger/vars.sh
/opt/intel/oneapi/mpi/2021.19/env/vars.sh
/opt/intel/oneapi/mpi/2021.19/etc/mpi/vars.sh
/opt/intel/oneapi/dpl/2022.14/env/vars.sh
/opt/intel/oneapi/dpl/2022.14/etc/dpl/vars.sh
/opt/intel/oneapi/tbb/2023.2/env/vars.sh
/opt/intel/oneapi/tbb/2023.2/etc/tbb/vars.sh
/opt/intel/oneapi/compiler/2026.2/env/vars.sh
/opt/intel/oneapi/compiler/2026.2/bin/icpx
/opt/intel/oneapi/compiler/2026.2/etc/compiler/vars.sh
/opt/intel/oneapi/tcm/1.6/env/vars.sh
/opt/intel/oneapi/dev-utilities/2026.2/env/vars.sh


In [ ]:
# Create a simple sycl_device_query.cpp to compile and test
cpp_code = """#include <sycl/sycl.hpp>
#include <iostream>

int main() {
    for (auto device : sycl::device::get_devices()) {
        std::cout << "Device: " << device.get_info<sycl::info::device::name>() << std::endl;
        std::cout << "  Type: " << (device.is_gpu() ? "GPU" : (device.is_cpu() ? "CPU" : "Host/Other")) << std::endl;
    }
    return 0;
}
"""

with open('sycl_device_query.cpp', 'w') as f:
    f.write(cpp_code)

print("[+] Created sycl_device_query.cpp")

[+] Created sycl_device_query.cpp


In [ ]:
import subprocess

# Source the correct environment and execute the compiled query binary
cmd = """bash -c 'source /opt/intel/oneapi/compiler/2026.2/env/vars.sh && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query'"""
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

print("--- SYCL Device Query Output ---")
if result.returncode == 0:
    print(result.stdout)
else:
    print("Error during execution:")
    print(result.stderr)

--- SYCL Device Query Output ---
Device:            Intel(R) Xeon(R) CPU @ 2.00GHz
  Type: CPU



In [ ]:
import subprocess

# Compile the SYCL code using the Intel C++ compiler (icpx)
compile_cmd = """bash -c 'source /opt/intel/oneapi/compiler/2026.2/env/vars.sh && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query'"""
result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)

print("--- Compilation & Execution Output ---")
if result.returncode == 0:
    print(result.stdout)
else:
    print("Error during compilation/execution:")
    print(result.stderr)

--- Compilation & Execution Output ---
Device:            Intel(R) Xeon(R) CPU @ 2.00GHz
  Type: CPU



In [ ]:
%%writefile vector_mul.cpp
// ==============================================================================
// Program: vector_mul.cpp
// Purpose: Perform parallel element-wise multiplication of two vectors using SYCL
// ==============================================================================
#include <sycl/sycl.hpp>
#include <iostream>
#include <vector>
#include <cmath>

int main() {
    const size_t N = 1000000; // 1 million elements
    std::vector<float> h_a(N, 2.0f);
    std::vector<float> h_b(N, 3.0f);
    std::vector<float> h_c(N, 0.0f);

    try {
        // Initialize SYCL queue with the default selector
        sycl::queue q{sycl::default_selector_v};
        std::cout << "[+] Running on device: "
                  << q.get_device().get_info<sycl::info::device::name>() << std::endl;

        // Create buffers to manage data transfer between host and device
        sycl::buffer<float, 1> buf_a(h_a.data(), sycl::range<1>(N));
        sycl::buffer<float, 1> buf_b(h_b.data(), sycl::range<1>(N));
        sycl::buffer<float, 1> buf_c(h_c.data(), sycl::range<1>(N));

        // Submit command group to the queue
        q.submit([&](sycl::handler& h) {
            // Accessors read and write to buffers
            sycl::accessor acc_a(buf_a, h, sycl::read_only);
            sycl::accessor acc_b(buf_b, h, sycl::read_only);
            sycl::accessor acc_c(buf_c, h, sycl::write_only, sycl::no_init);

            // Execute the kernel in parallel across N work-items
            h.parallel_for(sycl::range<1>(N), [=](sycl::id<1> idx) {
                acc_c[idx] = acc_a[idx] * acc_b[idx];
            });
        }).wait(); // Wait for execution to complete

    } catch (sycl::exception const& e) {
        std::cerr << "[-] SYCL exception caught: " << e.what() << std::endl;
        return 1;
    }

    // Verify the results on the host side
    bool correct = true;
    for (size_t i = 0; i < N; ++i) {
        if (std::abs(h_c[i] - 6.0f) > 1e-5f) {
            correct = false;
            break;
        }
    }

    if (correct) {
        std::cout << "[+] Verification PASSED! Element-wise multiplication computed successfully on " << N << " elements." << std::endl;
    } else {
        std::cout << "[-] Verification FAILED! Incorrect results calculated." << std::endl;
    }

    return 0;
}

Writing vector_mul.cpp


In [ ]:
import subprocess
import os

# 1. Reinstall Intel oneAPI if /opt/intel is empty
if not os.path.exists("/opt/intel/oneapi/compiler"):
    print("[*] Re-configuring and installing Intel oneAPI...")
    install_commands = '''
    set -e
    wget -qO - https://apt.repos.intel.com/intel-gpg-keys/GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB | gpg --dearmor | sudo tee /usr/share/keyrings/oneapi-archive-keyring.gpg > /dev/null
    echo "deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main" | sudo tee /etc/apt/sources.list.d/oneAPI.list
    sudo apt-get update -q
    sudo apt-get install -y -q intel-oneapi-compiler-dpcpp-cpp intel-oneapi-mpi-devel build-essential
    '''
    subprocess.run(install_commands, shell=True, executable='/bin/bash')

# 2. Check and run the application using the verified path
vars_path = "/opt/intel/oneapi/compiler/2026.2/env/vars.sh"
if os.path.exists(vars_path):
    print(f"[+] Sourcing environment from: {vars_path}")
    compile_cmd = f"""bash -c 'source {vars_path} && icpx -fsycl vector_mul.cpp -o vector_mul && ./vector_mul'"""
    result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)
    print("--- Vector Multiplication Output ---")
    if result.returncode == 0:
        print(result.stdout)
    else:
        print("Error during compilation/execution:")
        print(result.stderr)
else:
    print(f"[-] Error: {vars_path} was not found after installation attempts.")

[+] Sourcing environment from: /opt/intel/oneapi/compiler/2026.2/env/vars.sh
--- Vector Multiplication Output ---
[+] Running on device:            Intel(R) Xeon(R) CPU @ 2.00GHz
[+] Verification PASSED! Element-wise multiplication computed successfully on 1000000 elements.



In [ ]:
import os

# Let's locate the 'sycl/sycl.hpp' header file under /opt/intel/
print("Searching for sycl/sycl.hpp...")
sycl_headers = []
for root, dirs, files in os.walk("/opt/intel"):
    if "sycl.hpp" in files:
        sycl_headers.append(os.path.join(root, "sycl.hpp"))

if sycl_headers:
    print("[+] Found SYCL headers at:")
    for header in sycl_headers:
        print(header)
else:
    print("[-] SYCL header 'sycl.hpp' not found. Please ensure compiler is fully installed.")

Searching for sycl/sycl.hpp...
[+] Found SYCL headers at:
/opt/intel/oneapi/compiler/2026.2/include/CL/sycl.hpp
/opt/intel/oneapi/compiler/2026.2/include/sycl/sycl.hpp


In [ ]:
%%writefile sycl_device_query.cpp
// ==============================================================================
// Program: sycl_device_query.cpp
// Purpose: Introspect Intel SYCL platforms, compute devices, and memory capabilities
// ==============================================================================
#include <iostream>
#include <vector>
#include <string>

#if __has_include(<sycl/sycl.hpp>)
#include <sycl/sycl.hpp>
#define HAS_SYCL 1
#elif __has_include(<CL/sycl.hpp>)
#include <CL/sycl.hpp>
#define HAS_SYCL 1
#else
#define HAS_SYCL 0
#endif

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "      INTEL oneAPI / SYCL COMPUTE DEVICE DISCOVERY        " << std::endl;
    std::cout << "==========================================================" << std::endl;

#if HAS_SYCL
    // Enumerate all platforms and filter for those containing "Intel"
    auto platforms = sycl::platform::get_platforms();
    std::vector<sycl::platform> intel_platforms;

    for (const auto& platform : platforms) {
        std::string platform_name = platform.get_info<sycl::info::platform::name>();
        if (platform_name.find("Intel") != std::string::npos) {
            intel_platforms.push_back(platform);
        }
    }

    std::cout << "[+] Found " << intel_platforms.size() << " Intel SYCL Platform(s):" << std::endl;

    for (size_t p = 0; p < intel_platforms.size(); ++p) {
        std::cout << "\n  Platform [" << p << "]: "
                  << intel_platforms[p].get_info<sycl::info::platform::name>()
                  << " (" << intel_platforms[p].get_info<sycl::info::platform::vendor>() << ")"
                  << std::endl;

        auto devices = intel_platforms[p].get_devices();
        for (size_t d = 0; d < devices.size(); ++d) {
            std::cout << "    └── Device [" << d << "]: "
                      << devices[d].get_info<sycl::info::device::name>() << std::endl;
            std::cout << "        • Compute Units     : "
                      << devices[d].get_info<sycl::info::device::max_compute_units>() << std::endl;
            std::cout << "        • Max Work-Group Size: "
                      << devices[d].get_info<sycl::info::device::max_work_group_size>() << std::endl;
            std::cout << "        • Global Memory     : "
                      << (devices[d].get_info<sycl::info::device::global_mem_size>() / (1024 * 1024))
                      << " MB" << std::endl;
        }
    }
#else
    std::cout << "[-] SYCL headers not found. Running Native C++20 Multicore Host Inspector:" << std::endl;
    std::cout << "    • Host Architecture: x86_64 Multicore Processor" << std::endl;
    std::cout << "    • Execution Target : Host CPU SIMD Vector Engine" << std::endl;
#endif

    std::cout << "==========================================================" << std::endl;
    return 0;
}

Overwriting sycl_device_query.cpp


In [ ]:
%%writefile sycl_gemm.cpp
// ==============================================================================
// Program: sycl_gemm.cpp
// Purpose: 2D Matrix Multiplication in DPC++/SYCL with 2D Range Work-Items
// ==============================================================================
#include <iostream>
#include <vector>
#include <chrono>

constexpr size_t MATRIX_DIM = 1024; // 1024 x 1024 Matrix

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "   PARALLEL MATRIX MULTIPLICATION (1024 x 1024)           " << std::endl;
    std::cout << "==========================================================" << std::endl;

    size_t total_elements = MATRIX_DIM * MATRIX_DIM;
    std::vector<float> A(total_elements, 1.5f);
    std::vector<float> B(total_elements, 2.0f);
    std::vector<float> C(total_elements, 0.0f);

    auto start = std::chrono::high_resolution_clock::now();

    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < MATRIX_DIM; ++i) {
        for (size_t j = 0; j < MATRIX_DIM; ++j) {
            float sum = 0.0f;
            for (size_t k = 0; k < MATRIX_DIM; ++k) {
                sum += A[i * MATRIX_DIM + k] * B[k * MATRIX_DIM + j];
            }
            C[i * MATRIX_DIM + j] = sum;
        }
    }

    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration = end - start;

    double total_flops = 2.0 * MATRIX_DIM * MATRIX_DIM * MATRIX_DIM;
    double gflops = (total_flops * 1e-9) / (duration.count() * 1e-3);

    std::cout << "[+] Matrix Dimensions : " << MATRIX_DIM << " x " << MATRIX_DIM << std::endl;
    std::cout << "[+] Execution Time    : " << duration.count() << " ms" << std::endl;
    std::cout << "[+] Compute Speed     : " << gflops << " GFLOPs" << std::endl;
    std::cout << "[+] Corner Element C[0]: " << C[0] << " (Expected: " << (MATRIX_DIM * 1.5f * 2.0f) << ")" << std::endl;
    std::cout << "==========================================================" << std::endl;
    return 0;
}

Overwriting sycl_gemm.cpp


In [ ]:
%%writefile sycl_gemm.cpp
// ==============================================================================
// Program: sycl_gemm.cpp
// Purpose: 2D Matrix Multiplication in DPC++/SYCL with 2D Range Work-Items
// ==============================================================================
#include <sycl/sycl.hpp>
#include <iostream>
#include <vector>
#include <chrono>
#include <cmath>

constexpr size_t MATRIX_DIM = 1024; // 1024 x 1024 Matrix

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "   PARALLEL SYCL MATRIX MULTIPLICATION (1024 x 1024)      " << std::endl;
    std::cout << "==========================================================" << std::endl;

    size_t total_elements = MATRIX_DIM * MATRIX_DIM;
    std::vector<float> h_A(total_elements, 1.5f);
    std::vector<float> h_B(total_elements, 2.0f);
    std::vector<float> h_C(total_elements, 0.0f);

    auto start = std::chrono::high_resolution_clock::now();

    try {
        // Create SYCL queue with the default selector
        sycl::queue q{sycl::default_selector_v};
        std::cout << "[+] Running on device: "
                  << q.get_device().get_info<sycl::info::device::name>() << std::endl;

        // Define 2D ranges for the buffers
        sycl::range<2> matrix_range(MATRIX_DIM, MATRIX_DIM);

        // Create buffers
        sycl::buffer<float, 2> buf_A(h_A.data(), matrix_range);
        sycl::buffer<float, 2> buf_B(h_B.data(), matrix_range);
        sycl::buffer<float, 2> buf_C(h_C.data(), matrix_range);

        // Submit job to queue
        q.submit([&](sycl::handler& h) {
            // Create accessors
            sycl::accessor acc_A(buf_A, h, sycl::read_only);
            sycl::accessor acc_B(buf_B, h, sycl::read_only);
            sycl::accessor acc_C(buf_C, h, sycl::write_only, sycl::no_init);

            // Execute the 2D parallel kernel
            h.parallel_for(matrix_range, [=](sycl::id<2> idx) {
                size_t row = idx[0];
                size_t col = idx[1];
                float sum = 0.0f;
                for (size_t k = 0; k < MATRIX_DIM; ++k) {
                    sum += acc_A[row][k] * acc_B[k][col];
                }
                acc_C[row][col] = sum;
            });
        }).wait();

    } catch (sycl::exception const& e) {
        std::cerr << "[-] SYCL exception caught: " << e.what() << std::endl;
        return 1;
    }

    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration = end - start;

    double total_flops = 2.0 * MATRIX_DIM * MATRIX_DIM * MATRIX_DIM;
    double gflops = (total_flops * 1e-9) / (duration.count() * 1e-3);

    std::cout << "[+] Matrix Dimensions : " << MATRIX_DIM << " x " << MATRIX_DIM << std::endl;
    std::cout << "[+] Execution Time    : " << duration.count() << " ms" << std::endl;
    std::cout << "[+] Compute Speed     : " << gflops << " GFLOPs" << std::endl;
    std::cout << "[+] Corner Element C[0]: " << h_C[0] << " (Expected: " << (MATRIX_DIM * 1.5f * 2.0f) << ")" << std::endl;
    std::cout << "==========================================================" << std::endl;
    return 0;
}

Overwriting sycl_gemm.cpp


In [ ]:
import subprocess

vars_path = "/opt/intel/oneapi/compiler/2026.2/env/vars.sh"
compile_cmd = f"""bash -c 'source {vars_path} && icpx -fsycl sycl_gemm.cpp -o sycl_gemm && ./sycl_gemm'"""
result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)

print("--- SYCL GEMM Compilation & Execution Output ---")
if result.returncode == 0:
    print(result.stdout)
else:
    print("Error during compilation/execution:")
    print(result.stderr)

--- SYCL GEMM Compilation & Execution Output ---
   PARALLEL SYCL MATRIX MULTIPLICATION (1024 x 1024)      
[+] Running on device:            Intel(R) Xeon(R) CPU @ 2.00GHz
[+] Matrix Dimensions : 1024 x 1024
[+] Execution Time    : 1043.99 ms
[+] Compute Speed     : 2.057 GFLOPs
[+] Corner Element C[0]: 3072 (Expected: 3072)



In [ ]:
!g++ -std=c++20 -O3 -fopenmp -march=native sycl_gemm.cpp -o sycl_gemm && ./sycl_gemm


sycl_gemm.cpp:5:10: fatal error: sycl/sycl.hpp: No such file or directory
    5 | #include <sycl/sycl.hpp>
      |          ^~~~~~~~~~~~~~~
compilation terminated.


In [ ]:
%%writefile matrix_transpose.cpp
// ==============================================================================
// Program: matrix_transpose.cpp
// Purpose: Task C: Transposed Matrix Initialization with CPU/OMP Verification
// ==============================================================================
#include <iostream>
#include <vector>
#include <chrono>
#include <omp.h>
#include <cmath>

constexpr size_t DIM = 512; // 512 x 512 Matrix

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "      TRANSPOSED MATRIX INITIALIZATION & VERIFICATION     " << std::endl;
    std::cout << "==========================================================" << std::endl;

    std::vector<float> A(DIM * DIM, 0.0f);
    std::vector<float> T(DIM * DIM, 0.0f);

    // 1. Initialize original matrix A where A(i, j) = i * j
    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            A[i * DIM + j] = static_cast<float>(i * j);
        }
    }

    auto start = std::chrono::high_resolution_clock::now();

    // 2. Compute transpose matrix T where T(j, i) = A(i, j)
    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            T[j * DIM + i] = A[i * DIM + j];
        }
    }

    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration = end - start;

    // 3. Verify that the transpose matches our target mathematical values (T(j, i) == j * i)
    bool verified = true;
    #pragma omp parallel for reduction(&& : verified)
    for (size_t j = 0; j < DIM; ++j) {
        for (size_t i = 0; i < DIM; ++i) {
            float expected = static_cast<float>(j * i);
            if (std::abs(T[j * DIM + i] - expected) > 1e-5f) {
                verified = false;
            }
        }
    }

    std::cout << "[+] Matrix Dimensions : " << DIM << " x " << DIM << std::endl;
    std::cout << "[+] Transpose Time    : " << duration.count() << " ms" << std::endl;
    if (verified) {
        std::cout << "[+] Verification PASSED! Transposed matrix matches target mathematical values." << std::endl;
    } else {
        std::cout << "[-] Verification FAILED! Transposed matrix does not match expected values." << std::endl;
    }
    std::cout << "==========================================================" << std::endl;
    return 0;
}

Overwriting matrix_transpose.cpp


In [ ]:
import subprocess

# Compile the code using GCC with OpenMP enabled and execute it
compile_cmd = "g++ -O3 -fopenmp matrix_transpose.cpp -o matrix_transpose && ./matrix_transpose"
result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)

print("--- CPU/OMP Matrix Transpose Output ---")
if result.returncode == 0:
    print(result.stdout)
else:
    print("Error during compilation/execution:")
    print(result.stderr)

--- CPU/OMP Matrix Transpose Output ---
      TRANSPOSED MATRIX INITIALIZATION & VERIFICATION     
[+] Matrix Dimensions : 512 x 512
[+] Transpose Time    : 0.741694 ms
[+] Verification PASSED! Transposed matrix matches target mathematical values.



In [ ]:
%%writefile matrix_transpose_comparison.cpp
// ==============================================================================
// Program: matrix_transpose_comparison.cpp
// Purpose: Compare sequential vs OpenMP parallel matrix transpose execution times
// ==============================================================================
#include <iostream>
#include <vector>
#include <chrono>
#include <omp.h>
#include <cmath>

constexpr size_t DIM = 512;

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "    MATRIX TRANSPOSE: SEQUENTIAL VS PARALLEL PERFORMANCE  " << std::endl;
    std::cout << "==========================================================" << std::endl;

    std::vector<float> A(DIM * DIM, 0.0f);
    std::vector<float> T_seq(DIM * DIM, 0.0f);
    std::vector<float> T_par(DIM * DIM, 0.0f);

    // Initialize original matrix A with a simple pattern
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            A[i * DIM + j] = static_cast<float>(i * j);
        }
    }

    // 1. Sequential Transpose (No Parallelization)
    auto start_seq = std::chrono::high_resolution_clock::now();
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            T_seq[j * DIM + i] = A[i * DIM + j];
        }
    }
    auto end_seq = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration_seq = end_seq - start_seq;

    // 2. Parallel Transpose (With OpenMP Multi-threading)
    auto start_par = std::chrono::high_resolution_clock::now();
    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            T_par[j * DIM + i] = A[i * DIM + j];
        }
    }
    auto end_par = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration_par = end_par - start_par;

    // 3. Verification & Metrics
    bool match = true;
    for (size_t i = 0; i < DIM * DIM; ++i) {
        if (T_seq[i] != T_par[i]) {
            match = false;
            break;
        }
    }

    double speedup = duration_seq.count() / duration_par.count();

    std::cout << "[+] Matrix Size         : " << DIM << " x " << DIM << std::endl;
    std::cout << "[+] Sequential Time     : " << duration_seq.count() << " ms" << std::endl;
    std::cout << "[+] Parallel (OMP) Time : " << duration_par.count() << " ms" << std::endl;
    std::cout << "[+] Measured Speedup    : " << speedup << "x" << std::endl;
    std::cout << "[+] Verification Result : " << (match ? "PASSED (Outputs Match)" : "FAILED (Outputs Differ)") << std::endl;
    std::cout << "==========================================================" << std::endl;

    return 0;
}

Writing matrix_transpose_comparison.cpp


In [ ]:
import subprocess

# Compile and run the comparison program using GCC with OpenMP enabled
compile_cmd = "g++ -O3 -fopenmp matrix_transpose_comparison.cpp -o matrix_transpose_comparison && ./matrix_transpose_comparison"
result = subprocess.run(compile_cmd, shell=True, capture_output=True, text=True)

print("--- Sequential vs. Parallel Transpose Comparison ---")
if result.returncode == 0:
    print(result.stdout)
else:
    print("Error during compilation/execution:")
    print(result.stderr)

--- Sequential vs. Parallel Transpose Comparison ---
    MATRIX TRANSPOSE: SEQUENTIAL VS PARALLEL PERFORMANCE  
[+] Matrix Size         : 512 x 512
[+] Sequential Time     : 1.00804 ms
[+] Parallel (OMP) Time : 0.811014 ms
[+] Measured Speedup    : 1.24293x
[+] Verification Result : PASSED (Outputs Match)

